# 01 · Conocer los datos

Vamos a revisar las imágenes de células sanguíneas, contar las clases y preparar
los datos para entrenar. Autor: [nombre y apellidos].

## Preparación

Sube la carpeta `PC3_Deep_Learning` a **Mi unidad** de Google Drive, incluyendo
`data/raw/` con las imágenes descargadas. No hace falta subir `.venv`.
Usaremos esa carpeta en los tres notebooks. Ejecútalos en orden: **01 → 02 → 03**.
Para los notebooks 02 y 03, activa una GPU en Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/PC3_Deep_Learning
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as MostrarImagen
from src.utils import seed_everything

seed_everything(42)

## Separar entrenamiento y validación

Las imágenes ya están en `data/raw/images/`. Reservamos el 20% de TRAIN para
validación y dejamos TEST como viene. Usamos la semilla 42 para repetir la misma
separación. Las imágenes idénticas de TRAIN se mantienen juntas.

In [ ]:
from src.utils import make_manifest, save_figure, CLASSES
from PIL import Image
import os

make_manifest("data/raw", "data/splits", seed=42, validation_fraction=0.2)
datos = pd.read_csv("data/splits/manifest.csv")
os.makedirs("results/tables", exist_ok=True)
display(datos.head())

## Cantidad de imágenes por clase

Esta tabla muestra cuántas imágenes tenemos para entrenar, validar y evaluar.

In [ ]:
cantidades = datos.groupby(["class", "split"]).size().unstack()
cantidades = cantidades.loc[CLASSES, ["train", "val", "test"]]
cantidades.to_csv("results/tables/class_distribution.csv")
display(cantidades)

cantidades.plot.bar(figsize=(10, 4), rot=15)
plt.title("Imágenes por clase")
plt.xlabel("Tipo de célula")
plt.ylabel("Cantidad de imágenes")
plt.legend(title="Partición")
save_figure(plt.gcf(), "results/figures/eda_class_distribution.png")
display(MostrarImagen(filename="results/figures/eda_class_distribution.png"))

## Ejemplos de las cuatro clases

Mostramos cuatro imágenes de entrenamiento de cada clase.

In [ ]:
fig, ejes = plt.subplots(4, 4, figsize=(10, 9))
for fila, clase in enumerate(CLASSES):
    ejemplos = datos[(datos["class"] == clase) & (datos["split"] == "train")]
    ejemplos = ejemplos.sample(4, random_state=42)
    for columna, archivo in enumerate(ejemplos["path"]):
        with Image.open("data/raw/images/" + archivo) as imagen:
            ejes[fila, columna].imshow(imagen.convert("RGB"))
        ejes[fila, columna].set_title(clase, fontsize=9)
        ejes[fila, columna].axis("off")
save_figure(fig, "results/figures/eda_examples.png")
display(MostrarImagen(filename="results/figures/eda_examples.png"))

## Tamaño de las imágenes

In [ ]:
tamanos = datos.groupby(["width", "height"]).size().reset_index(name="cantidad")
tamanos.to_csv("results/tables/image_dimensions.csv", index=False)
display(tamanos)

plt.figure(figsize=(7, 4))
plt.bar(tamanos["width"].astype(str) + " × " + tamanos["height"].astype(str), tamanos["cantidad"])
plt.title("Tamaño de las imágenes")
plt.xlabel("Ancho × alto en píxeles")
plt.ylabel("Cantidad")
save_figure(plt.gcf(), "results/figures/eda_dimensions.png")
display(MostrarImagen(filename="results/figures/eda_dimensions.png"))

## Imágenes repetidas

Revisamos si hay imágenes idénticas. Esta revisión no detecta fotos rotadas ni
permite saber si dos imágenes pertenecen al mismo paciente. Eso limita la evaluación.

In [ ]:
with open("data/splits/audit.json") as archivo:
    revision = json.load(archivo)
display(revision)
display(datos[datos.duplicated("sha256", keep=False)][["path", "class", "split"]].head(20))

## Preparación de las imágenes

LeNet y VGG usan imágenes de 64×64. Durante el entrenamiento se aplican giros
y volteos. Validación y test no usan esos cambios. ResNet usará 224×224 y
la normalización de ImageNet. Al cambiar el tamaño se modifica la proporción original.

In [ ]:
from src.utils import make_loaders
from src.train import DEFAULT_SETTINGS, experiment_plan

cargadores = make_loaders(
    "data/raw/images", "data/splits/manifest.csv",
    experiment_plan(DEFAULT_SETTINGS, "scratch")[0]
)
imagenes, etiquetas = next(iter(cargadores["train"]))

fig, ejes = plt.subplots(2, 4, figsize=(10, 5))
for eje, imagen, etiqueta in zip(ejes.flat, imagenes[:8], etiquetas[:8]):
    eje.imshow((imagen.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1))
    eje.set_title(CLASSES[int(etiqueta)], fontsize=9)
    eje.axis("off")
save_figure(fig, "results/figures/eda_augmented_batch.png")
display(MostrarImagen(filename="results/figures/eda_augmented_batch.png"))

## Comentarios

- ¿Hay una cantidad parecida de imágenes por clase?
- ¿Qué diferencias de color, forma o fondo se observan?
- ¿Hay imágenes repetidas? ¿Cómo podrían afectar los resultados?

**Respuesta:** [completar después de ejecutar].

Continúa con `02_lenet_vgg.ipynb`.